# Занятие 36. Практика: бустинг в реальных библиотеках

**Тема:** предсказание прогрессирования диабета по медицинским признакам (реальный датасет `load_diabetes`).  
**Модельный фокус:** градиентный бустинг для регрессии.  
**Библиотеки:** `sklearn`, `xgboost`, `lightgbm`, `catboost` доступны в учебной среде и сравниваются обязательно.

## Легенда

Вы работаете в аналитической команде медицинского сервиса. Нужно предсказывать численный индекс прогрессирования заболевания через год, чтобы врачам было проще выделять пациентов с высоким риском.

Почему это хороший кейс для бустинга:
- задача регрессии на табличных данных;
- признаки имеют нелинейные связи;
- важны качество, скорость обучения и интерпретация;
- в реальных проектах обычно сравнивают несколько библиотек бустинга, а не одну.

## Что нужно сделать

1. Подготовить данные и baseline.
2. Разделить данные на train / validation / test.
3. Обучить и сравнить `GradientBoostingRegressor` и `HistGradientBoostingRegressor` из `sklearn`.
4. Обязательно сравнить внешние библиотеки (`xgboost`, `lightgbm`, `catboost`).
5. Выбрать модель по validation, а test использовать один раз — для финальной проверки выбранного решения.
6. Сформулировать практические выводы: когда какую библиотеку выбирать, какие у неё плюсы и минусы.

### Оценивание (30 баллов)

| № | Тема | Баллы |
|---|------|------:|
| 0 | Загрузка и split | 3 |
| 1 | Baseline и метрики | 3 |
| 2 | Sklearn GradientBoostingRegressor | 4 |
| 3 | Подбор гиперпараметров для GBDT | 5 |
| 4 | HistGradientBoostingRegressor и early stopping | 4 |
| 5 | Внешние библиотеки бустинга | 5 |
| 6 | Единый рейтинг моделей | 2 |
| 7 | Преимущества и недостатки библиотек | 2 |
| 8 | Финальный инженерный вывод | 2 |
| | **Итого** | **30** |

---
## Среда и библиотеки

В учебной среде `xgboost`, `lightgbm` и `catboost` уже установлены. Если вы запускаете блокнот дома и импорт ниже падает, установите их отдельно перед занятием.

In [ ]:
# В учебной среде установка не нужна.
# Если запускаете дома и библиотек нет, выполните в терминале:
# pip install xgboost lightgbm catboost


In [ ]:
import importlib
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor

RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.25  # 25% от train_val ≈ 20% от всех данных


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def evaluate_regression(model_name, y_true, y_pred):
    return {
        "model": model_name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }


---
## Задание 0. Загрузка и split — **3 балла**

Загрузите датасет `load_diabetes(as_frame=True)` и соберите матрицу признаков `X` и целевую переменную `y`. Сначала отделите финальный test, а из оставшейся части выделите validation: test понадобится только один раз в конце, когда модель уже выбрана.

После разбиения выведите размеры train, validation и test, а также первые 5 строк `X_train`. Это быстрая проверка, что данные загружены и дальше в ноутбуке используются правильные выборки.

---
## Задание 1. Baseline и метрики — **3 балла**

Постройте простой baseline: обучите `DummyRegressor(strategy='mean')` на train. Такая модель всегда предсказывает среднее значение цели и нужна как нижняя планка качества.

Посчитайте MAE, RMSE и R2 на validation и сохраните результат в `baseline_metrics`. Дальше все модели бустинга нужно сравнивать с этим baseline, чтобы было понятно, дают ли они реальное улучшение.

---
## Задание 2. Sklearn GradientBoostingRegressor — **4 балла**

Обучите базовую модель `GradientBoostingRegressor(random_state=RANDOM_STATE)` на train и оцените её на validation. Посчитайте те же метрики, что и для baseline: MAE, RMSE и R2.

Соберите сравнение с baseline в единую таблицу и напишите короткий вывод: стала ли модель лучше, по каким метрикам это видно и насколько велико улучшение.

---
## Задание 3. Подбор гиперпараметров для GBDT — **5 баллов**

Настройте `GridSearchCV` для `GradientBoostingRegressor`. В сетке подбора обязательно проверьте несколько значений `n_estimators`, `learning_rate` и `max_depth`: эти параметры управляют числом деревьев, скоростью исправления ошибок и сложностью отдельных деревьев.

Используйте `scoring='neg_mean_absolute_error'`. В sklearn MAE хранится со знаком минус, потому что при подборе большее значение считается лучшим, поэтому при выводе результата переведите его обратно в обычный положительный MAE. После подбора выведите `best_params_`, CV-метрику и качество лучшей модели на validation.

---
## Задание 4. HistGradientBoostingRegressor и early stopping — **4 балла**

Обучите `HistGradientBoostingRegressor` и включите `early_stopping=True`. Ранняя остановка нужна, чтобы модель прекращала обучение, когда новые итерации перестают заметно улучшать качество.

Замерьте время обучения, посчитайте метрики на validation и сравните результат с лучшей моделью из задания 3. В сравнении важно отразить не только качество, но и скорость: иногда чуть менее точная модель оказывается удобнее для рабочего сервиса.

---
## Задание 5. Внешние библиотеки бустинга — **5 баллов**

Сравните бустинг из внешних библиотек с моделями sklearn. Импортируйте `XGBRegressor`, `LGBMRegressor` и `CatBoostRegressor`, затем обучите по одной базовой модели XGBoost, LightGBM и CatBoost.

Для каждой модели посчитайте MAE, RMSE и R2 на validation и соберите результаты в общую таблицу `external_results`. Используйте спокойные базовые настройки и фиксируйте `random_state`, где это возможно, чтобы сравнение было воспроизводимым.

---
## Задание 6. Единый рейтинг моделей — **2 балла**

Соберите результаты baseline, sklearn-моделей и внешних библиотек в один DataFrame `all_results`. Отсортируйте таблицу по validation RMSE по возрастанию, чтобы лучшая модель оказалась сверху.

Постройте столбчатый график RMSE для всех моделей. График нужен не для красоты, а для быстрого сравнения: по нему должно быть понятно, какие модели близки по качеству, а какие заметно отстают.

---
## Задание 7. Преимущества и недостатки библиотек — **2 балла**

Заполните таблицу (markdown **или** `DataFrame`) минимум для 4 библиотек:
- `sklearn GradientBoosting`
- `sklearn HistGradientBoosting`
- `XGBoost`
- `LightGBM`
- `CatBoost`

Для каждой укажите:
1. что в библиотеке удобно;
2. что может мешать;
3. когда её разумно брать для рабочего сервиса, а когда лучше не брать.

---
## Задание 8. Финальный инженерный вывод — **2 балла**

Напишите 8–12 предложений:
- какая модель лучшая по validation;
- насколько она лучше baseline по validation RMSE/MAE;
- какую модель вы бы выбрали для финальной проверки на test;
- какую библиотеку взяли бы при ограничении по времени обучения;
- какую — при максимальном фокусе на качестве;
- какие риски нужно проверить перед использованием модели в рабочем сервисе.

После выбора модели по validation выполните **одну финальную оценку на test**.